# TinyImageNet Canonical ALiBi — All-in-One Runner

Single-notebook version that does everything from scratch in one session:
Drive mount → deps → stage scripts → patch → reset → download → extract → 
create variant script → train.

**To run two parallel sessions** (one for 1D-ALiBi, one for 2D-ALiBi):
open this notebook in two Colab sessions, change `PE_VARIANT` on top of the
**Setup** cell to `'1d'` in one and `'2d'` in the other, then run both cells
in each session. Each session is fully self-contained (no shared state in
`/content/`). Both write into the same Drive folder
(`Trained models_TinyImageNet_v2/`) into disjoint sub-folders, so the
results merge naturally when both finish.

**Prerequisites on Drive** (anywhere under MyDrive — the script searches):
- `apply_2d_alibi_patch.py`
- `full_scale_experiment.py` (unpatched)
- `tinyimagenet_alibi_canonical.py` (master training script)

Compute: ~22 h on G4 (RTX 6000 Blackwell). TinyImageNet download (~250 MB)
happens once per session into ephemeral `/content/`.

## Setup (one cell, end-to-end)

Set `PE_VARIANT` at the top. This is the only thing that differs between
the two parallel sessions. Cell aborts immediately and clearly if anything
is wrong, so you don't waste hours discovering a bad state.

In [ ]:
# ============================================================
# CONFIG -- only line that differs between parallel sessions
# ============================================================
PE_VARIANT = '1d'   # <<<<<<<<< CHANGE TO '2d' IN THE OTHER SESSION
assert PE_VARIANT in ('1d', '2d'), "PE_VARIANT must be '1d' or '2d'"

# ============================================================
# Step 1: Mount Drive + GPU check
# ============================================================
print('=' * 70)
print(f'TinyImageNet canonical ALiBi setup -- variant: {PE_VARIANT.upper()}')
print('=' * 70)

from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No GPU available -- aborting'
print(f'\nGPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Torch:  {torch.__version__}')

# ============================================================
# Step 2: Install dependencies
# ============================================================
print('\n--- Installing dependencies ---')
import subprocess
subprocess.run(['pip', 'install', '-q', 'timm', 'tqdm'], check=True)
print('  done')

# ============================================================
# Step 3: Stage scripts from Drive into /content/
# ============================================================
print('\n--- Staging scripts from Drive ---')
import os, shutil, sys

DRIVE_BASE = '/content/drive/MyDrive'
PE_EXPERIMENT_BASE = os.path.join(DRIVE_BASE, 'pe_experiment')

needed = [
    ('apply_2d_alibi_patch.py', [
        os.path.join(DRIVE_BASE, 'apply_2d_alibi_patch.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'apply_2d_alibi_patch.py'),
    ]),
    ('full_scale_experiment.py', [
        os.path.join(PE_EXPERIMENT_BASE, 'full_scale_experiment.py'),
        os.path.join(DRIVE_BASE, 'full_scale_experiment.py'),
    ]),
    ('tinyimagenet_alibi_canonical.py', [
        os.path.join(DRIVE_BASE, 'tinyimagenet_alibi_canonical.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'tinyimagenet_alibi_canonical.py'),
    ]),
]

for fname, candidates in needed:
    src = next((p for p in candidates if os.path.isfile(p)), None)
    if src is None:
        raise FileNotFoundError(
            f'\nCannot find {fname} in any of:\n  ' + '\n  '.join(candidates))
    shutil.copy(src, f'/content/{fname}')
    print(f'  {fname}  <-  {src}')

# ============================================================
# Step 4: Apply 2D-ALiBi patch (regenerate full_scale_experiment_v2.py)
# ============================================================
print('\n--- Applying 2D-ALiBi patch ---')
result = subprocess.run(
    ['python', 'apply_2d_alibi_patch.py',
     'full_scale_experiment.py', 'full_scale_experiment_v2.py'],
    cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'Patch failed with exit code {result.returncode}')
assert os.path.isfile('/content/full_scale_experiment_v2.py')
print('  patched module ready')

# ============================================================
# Step 5: Reset any broken cached state
# (does NOT touch Drive results -- only ephemeral /content/ and the
#  broken Drive data layout if it exists from earlier attempts)
# ============================================================
print('\n--- Cleaning broken cached state ---')
broken_drive = '/content/drive/MyDrive/tinyimagenet'
if os.path.isdir(broken_drive):
    train_dir = os.path.join(broken_drive, 'tiny-imagenet-200', 'train')
    n_train_classes = (
        len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
        if os.path.isdir(train_dir) else 0
    )
    if n_train_classes != 200:
        print(f'  Drive layout incomplete ({n_train_classes}/200 train classes) — removing')
        shutil.rmtree(broken_drive)
    else:
        print(f'  Drive layout looks complete ({n_train_classes} train classes) — leaving alone')

for f in ['/content/tin_run_1d.py', '/content/tin_run_2d.py']:
    if os.path.isfile(f):
        os.remove(f)
        print(f'  removed cached {f}')

if os.path.isdir('/content/tinyimagenet'):
    # Only nuke if it's not already a complete extraction
    train_dir = '/content/tinyimagenet/tiny-imagenet-200/train'
    n_train_classes = (
        len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
        if os.path.isdir(train_dir) else 0
    )
    if n_train_classes != 200:
        shutil.rmtree('/content/tinyimagenet')
        print(f'  removed incomplete local /content/tinyimagenet ({n_train_classes}/200)')

# ============================================================
# Step 6: Download + extract TinyImageNet to LOCAL /content/
# (faster than Drive for training I/O; no Drive race conditions)
# ============================================================
print('\n--- Preparing TinyImageNet (local) ---')
import urllib.request, zipfile

LOCAL_DATA = '/content/tinyimagenet'
os.makedirs(LOCAL_DATA, exist_ok=True)
zip_path  = os.path.join(LOCAL_DATA, 'tiny-imagenet-200.zip')
extracted = os.path.join(LOCAL_DATA, 'tiny-imagenet-200')

train_dir = os.path.join(extracted, 'train')
if (os.path.isdir(train_dir) and
        len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]) == 200):
    print(f'  already extracted at {extracted} — skipping download')
else:
    print(f'  downloading TinyImageNet (~250 MB) ...')
    urllib.request.urlretrieve(
        'http://cs231n.stanford.edu/tiny-imagenet-200.zip', zip_path)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f'    done ({size_mb:.1f} MB)')
    assert size_mb > 240, f'Zip too small ({size_mb:.1f} MB) — download failed'

    print(f'  extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    print(f'    done')

# Verify canonical TIN structure
for required in ['train', 'val', 'wnids.txt', 'words.txt']:
    path = os.path.join(extracted, required)
    assert os.path.exists(path), f'Missing required item: {path}'
val_ann = os.path.join(extracted, 'val', 'val_annotations.txt')
assert os.path.isfile(val_ann), f'Missing {val_ann}'

n_train_classes = len([d for d in os.listdir(train_dir)
                        if os.path.isdir(os.path.join(train_dir, d))])
n_val_images = len([f for f in os.listdir(os.path.join(extracted, 'val', 'images'))
                     if f.lower().endswith('.jpeg')])
print(f'  train classes: {n_train_classes}  (expected 200)')
print(f'  val images:    {n_val_images}  (expected 10000)')
assert n_train_classes == 200, f'Got {n_train_classes} train classes'
assert n_val_images == 10000, f'Got {n_val_images} val images'

# Reorganise val/ into ImageFolder layout (idempotent)
val_reorg_dir = os.path.join(extracted, 'val_reorganised')
if (not os.path.isdir(val_reorg_dir) or
        len(os.listdir(val_reorg_dir)) < 200):
    if os.path.isdir(val_reorg_dir):
        shutil.rmtree(val_reorg_dir)
    os.makedirs(val_reorg_dir)
    print(f'  reorganising val/ into ImageFolder layout ...')
    with open(val_ann) as f:
        for line in f:
            fname, cls = line.strip().split('\t')[:2]
            cls_dir = os.path.join(val_reorg_dir, cls)
            os.makedirs(cls_dir, exist_ok=True)
            src = os.path.join(extracted, 'val', 'images', fname)
            dst = os.path.join(cls_dir, fname)
            if os.path.isfile(src) and not os.path.isfile(dst):
                os.link(src, dst)
    n_val_classes = len(os.listdir(val_reorg_dir))
    assert n_val_classes == 200, f'Got {n_val_classes} val classes after reorg'
    print(f'    done — {n_val_classes} val classes')
else:
    print(f'  val_reorganised already prepared ({len(os.listdir(val_reorg_dir))} classes)')

# ============================================================
# Step 7: Create the per-variant training script
# (patches both PE_TYPES and DATA_DIR in the master)
# ============================================================
print('\n--- Creating variant training script ---')
with open('/content/tinyimagenet_alibi_canonical.py') as f:
    text = f.read()

# Patch 1: PE_TYPES
ORIGINAL_PE = "PE_TYPES = ['alibi', 'alibi_2d']"
REPLACEMENT_PE = ("PE_TYPES = ['alibi']" if PE_VARIANT == '1d'
                   else "PE_TYPES = ['alibi_2d']")
if ORIGINAL_PE not in text:
    raise RuntimeError(
        f'Cannot find PE_TYPES line in master script. Expected: {ORIGINAL_PE!r}')
text = text.replace(ORIGINAL_PE, REPLACEMENT_PE, 1)

# Patch 2: DATA_DIR -> local /content/
ORIGINAL_DATA = "DATA_DIR     = os.path.join(DRIVE_BASE, 'tinyimagenet')"
REPLACEMENT_DATA = "DATA_DIR     = '/content/tinyimagenet'"
if ORIGINAL_DATA not in text:
    raise RuntimeError(
        f'Cannot find DATA_DIR line in master script. Expected: {ORIGINAL_DATA!r}')
text = text.replace(ORIGINAL_DATA, REPLACEMENT_DATA, 1)

assert text.count('PE_TYPES =') == 1, 'Multiple PE_TYPES after patching'
assert text.count('DATA_DIR     =') == 1, 'Multiple DATA_DIR after patching'

dst = f'/content/tin_run_{PE_VARIANT}.py'
with open(dst, 'w') as f:
    f.write(text)
print(f'  wrote {dst}')
print(f'    PE_TYPES = {REPLACEMENT_PE.split("=")[1].strip()}')
print(f'    DATA_DIR = /content/tinyimagenet  (local)')

# ============================================================
# Final verification
# ============================================================
print('\n--- Final verification ---')
checks = {
    f'/content/tin_run_{PE_VARIANT}.py':         os.path.isfile(f'/content/tin_run_{PE_VARIANT}.py'),
    '/content/full_scale_experiment_v2.py':      os.path.isfile('/content/full_scale_experiment_v2.py'),
    '/content/tinyimagenet/tiny-imagenet-200/train': os.path.isdir('/content/tinyimagenet/tiny-imagenet-200/train'),
    '/content/tinyimagenet/tiny-imagenet-200/val_reorganised': os.path.isdir('/content/tinyimagenet/tiny-imagenet-200/val_reorganised'),
}
for path, ok in checks.items():
    marker = 'OK' if ok else 'MISSING'
    print(f'  [{marker:7s}] {path}')
if not all(checks.values()):
    raise RuntimeError('Some required files are missing -- check above')

print('\n' + '=' * 70)
print(f'Setup complete for variant {PE_VARIANT.upper()}.')
print(f'Ready to run the TRAIN cell.')
print('=' * 70)


## Train

Runs the variant-specific training script. Streams per-epoch output. If
the Colab session disconnects mid-training, you must re-run the Setup cell
again (because `/content/` is wiped), then re-launch this cell — but note
that the master script restarts a partial run from scratch (no epoch-level
resume). Only fully-completed runs are preserved on Drive.

In [ ]:
import subprocess, sys

script_path = f'/content/tin_run_{PE_VARIANT}.py'
assert __import__('os').path.isfile(script_path), \
    f'{script_path} missing — re-run the Setup cell first'

cmd = f'cd /content && python -u {script_path}'

print('=' * 70)
print(f'TinyImageNet canonical training -- variant: {PE_VARIANT.upper()}')
print(f'Script: {script_path}')
print('=' * 70)

p = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
for line in p.stdout:
    print(line, end='', flush=True)
    sys.stdout.flush()
exit_code = p.wait()

print('\n' + '=' * 70)
print('ALL DONE' if exit_code == 0 else f'Exited with code {exit_code}')
print('=' * 70)
